In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.utils import resample

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

from scipy import stats
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.stats import f_oneway, chi2_contingency
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit

import matplotlib.patches as patches

In [2]:
df = pd.read_excel('covid_mental_health.xlsx')


columns_to_drop = [
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_others',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_others'
]
data1 = df.drop(columns=columns_to_drop)


columns_to_select = [
    'birth_year', 'country_ofResidence', 'Level_ofEducation', 'current_work_status',
    'do_youHave_medical_insurance', 'current_relationship_status', 'genderAt_birth',
    'pre_birth_year', 'pre_country_ofResidence', 'pre_Level_ofEducation', 'pre_current_work_status',
    'pre_do_youHave_medical_insurance', 'pre_current_relationship_status', 'pre_genderAt_birth',
    'having_difficulty_concentrating', 'feeling_irretablee_orHaving_angry_outburst',
    'trouble_falling_orStaying_asSleep', 'feeling_asIfYourFeature_willBer_cut_short',
    'feeling_emotional_numb_orBeeing_unble_toLoving_feeling_forThose_closeTo_you',
    'feeling_distant_from_other_people', 'loss_ofInterest_activity_that_you_enjoyed',
    'trouble_memembering_important_part_ofStressful_experience_from_past',
    'avoiding_activy_because_they_remind_you_stressful_experience_from_past',
    'avoiding_thinking_about_stressful_experience_from_past',
    'having_physical_reaction_when_somthing_remind_you_stressful_experience',
    'feeling_upset_when_something_remind_you_ofStressful_experience_fromPast',
    'suddenly_feel_asIf_aStressfull_experience_were_happening_again',
    'repeated_disturbing_dream_ofStressfull_experiencce_from_past',
    'repeated_disturbing_memory_though_image_ofStressfull_experiencce_from_past',
    'pre_having_difficulty_concentrating', 'pre_feeling_irretablee_orHaving_angry_outburst',
    'pre_trouble_falling_orStaying_asSleep', 'pre_feeling_asIfYourFeature_willBer_cut_short',
    'pre_feeling_emotional_numb_orBeeing_unble_toLoving_feeling_forThose_closeTo_you',
    'pre_feeling_distant_from_other_people', 'pre_trouble_memembering_important_part_ofStressful_experience_from_past',
    'pre_loss_ofInterest_activity_that_you_enjoyed',
    'pre_avoiding_activy_because_they_remind_you_stressful_experience_from_past',
    'pre_feeling_upset_when_something_remind_you_ofStressful_experience_fromPast',
    'pre_avoiding_thinking_about_stressful_experience_from_past',
    'pre_having_physical_reaction_when_somthing_remind_you_stressful_experience',
    'pre_suddenly_feel_asIf_aStressfull_experience_were_happening_again',
    'pre_repeated_disturbing_dream_ofStressfull_experiencce_from_past',
    'pre_repeated_disturbing_memory_though_image_ofStressfull_experiencce_from_past',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_onPhone',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_video_chat',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_inPerson',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_spent_time_with_pet',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_meditation',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_inHome',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_outdoor',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_gardening',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_creative_activity',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_learn_new_skill',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taking_breaks_from_the_news',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_other',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_onPhone',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_video_chat',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_inPerson',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_spent_time_with_pet',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_meditation',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_inHome',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_outdoor',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_gardening',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_creative_activity',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_learn_new_skill',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taking_breaks_from_the_news',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_other',
    'what_you_experiencing_during_covid_period_diagnosed_covid_positive',
    'what_you_experiencing_during_covid_period_fear_ofGetting_covid',
    'what_you_experiencing_during_covid_period_fear_ofGiving_covid',
    'what_you_experiencing_during_covid_period_worrying_about_people_otherThanMe',
    'what_you_experiencing_during_covid_period_stigma_from_other_people',
    'what_you_experiencing_during_covid_period_frustration',
    'what_you_experiencing_during_covid_period_anxiety',
    'what_you_experiencing_during_covid_period_depression',
    'what_you_experiencing_during_covid_period_loneliness',
    'what_you_experiencing_during_covid_period_anger',
    'what_you_experiencing_during_covid_period_grief_ofFeeling_ofLoss',
    'what_you_experiencing_during_covid_period_changing_sleep_patern',
    'what_you_experiencing_during_covid_period_notGetting_exercise',
    'what_you_experiencing_during_covid_period_notGetting_enough_emotionOrSocial_support',
    'what_you_experiencing_during_covid_period_notGetting_enough_financial_support',
    'what_you_experiencing_during_covid_period_confusion_about_covid',
    'what_you_experiencing_during_covid_period_confusion_about_where_toGet_info_about_covid',
    'what_you_experiencing_during_covid_period_difficulty_obtaining_mask',
    'what_you_experiencing_during_covid_period_difficulty_washing_hand',
    'what_you_experiencing_during_covid_period_none_ofThe_avobe',
    'pre_what_you_experiencing_during_covid_period_diagnosed_covid_positive',
    'pre_what_you_experiencing_during_covid_period_fear_ofGetting_covid',
    'pre_what_you_experiencing_during_covid_period_fear_ofGiving_covid',
    'pre_what_you_experiencing_during_covid_period_worrying_about_people_otherThanMe',
    'pre_what_you_experiencing_during_covid_period_stigma_from_other_people',
    'pre_what_you_experiencing_during_covid_period_frustration',
    'pre_what_you_experiencing_during_covid_period_anxiety',
    'pre_what_you_experiencing_during_covid_period_depression',
    'pre_what_you_experiencing_during_covid_period_loneliness',
    'pre_what_you_experiencing_during_covid_period_anger',
    'pre_what_you_experiencing_during_covid_period_grief_ofFeeling_ofLoss',
    'pre_what_you_experiencing_during_covid_period_changing_sleep_patern',
    'pre_what_you_experiencing_during_covid_period_notGetting_exercise',
    'pre_what_you_experiencing_during_covid_period_notGetting_enough_emotionOrSocial_support',
    'pre_what_you_experiencing_during_covid_period_notGetting_enough_financial_support',
    'pre_what_you_experiencing_during_covid_period_confusion_about_covid',
    'pre_what_you_experiencing_during_covid_period_confusion_about_where_toGet_info_about_covid',
    'pre_what_you_experiencing_during_covid_period_difficulty_obtaining_mask',
    'pre_what_you_experiencing_during_covid_period_difficulty_washing_hand',
    'pre_what_you_experiencing_during_covid_period_none_ofThe_avobe',
    'what_you_experiencing_during_covid_period_anxiety',
    'what_you_experiencing_during_covid_period_depression',
    'what_you_experiencing_during_covid_period_loneliness',
    'what_you_experiencing_during_covid_period_anger',
    'what_you_experiencing_during_covid_period_frustration',
    'what_you_experiencing_during_covid_period_grief_ofFeeling_ofLoss',
    'what_you_experiencing_during_covid_period_changing_sleep_patern',
    'what_you_experiencing_during_covid_period_notGetting_exercise',
    'what_you_experiencing_during_covid_period_fear_ofGetting_covid',
    'what_you_experiencing_during_covid_period_fear_ofGiving_covid',
    'what_you_experiencing_during_covid_period_worrying_about_people_otherThanMe',
    'what_you_experiencing_during_covid_period_stigma_from_other_people',
    'what_you_experiencing_during_covid_period_notGetting_enough_emotionOrSocial_support'
]

data = data1[columns_to_select]


pre_columns = [col for col in columns_to_select if col.startswith('pre_')]
post_columns = [col for col in columns_to_select if not col.startswith('pre_')]

data_pre = data[pre_columns]
data_post = data[post_columns].loc[:, ~data[post_columns].columns.duplicated()]

In [4]:
all_18_items = [
    'pre_what_you_experiencing_during_covid_period_anxiety',
    'pre_what_you_experiencing_during_covid_period_depression',
    'pre_what_you_experiencing_during_covid_period_loneliness',
    'pre_what_you_experiencing_during_covid_period_anger',
    'pre_what_you_experiencing_during_covid_period_frustration',
    'pre_what_you_experiencing_during_covid_period_grief_ofFeeling_ofLoss',
    'pre_what_you_experiencing_during_covid_period_changing_sleep_patern',
    'pre_what_you_experiencing_during_covid_period_notGetting_exercise',
    'pre_what_you_experiencing_during_covid_period_fear_ofGetting_covid',
    'pre_what_you_experiencing_during_covid_period_fear_ofGiving_covid',
    'pre_what_you_experiencing_during_covid_period_worrying_about_people_otherThanMe',
    'pre_what_you_experiencing_during_covid_period_stigma_from_other_people',
    'pre_what_you_experiencing_during_covid_period_notGetting_enough_financial_support',
    'pre_what_you_experiencing_during_covid_period_confusion_about_covid',
    'pre_what_you_experiencing_during_covid_period_confusion_about_where_toGet_info_about_covid',
    'pre_what_you_experiencing_during_covid_period_notGetting_enough_emotionOrSocial_support',
    'pre_what_you_experiencing_during_covid_period_difficulty_obtaining_mask',
    'pre_what_you_experiencing_during_covid_period_difficulty_washing_hand'
]


final_15_items = [
    'pre_what_you_experiencing_during_covid_period_anxiety',
    'pre_what_you_experiencing_during_covid_period_depression',
    'pre_what_you_experiencing_during_covid_period_loneliness',
    'pre_what_you_experiencing_during_covid_period_anger',
    'pre_what_you_experiencing_during_covid_period_frustration',
    'pre_what_you_experiencing_during_covid_period_grief_ofFeeling_ofLoss',
    'pre_what_you_experiencing_during_covid_period_changing_sleep_patern',
    'pre_what_you_experiencing_during_covid_period_notGetting_exercise',
    'pre_what_you_experiencing_during_covid_period_fear_ofGetting_covid',
    'pre_what_you_experiencing_during_covid_period_fear_ofGiving_covid',
    'pre_what_you_experiencing_during_covid_period_worrying_about_people_otherThanMe',
    'pre_what_you_experiencing_during_covid_period_stigma_from_other_people',
    'pre_what_you_experiencing_during_covid_period_notGetting_enough_emotionOrSocial_support',
    'pre_what_you_experiencing_during_covid_period_confusion_about_covid',
    'pre_what_you_experiencing_during_covid_period_confusion_about_where_toGet_info_about_covid'
]

data_18 = data_pre[all_18_items].dropna()
data_15 = data_pre[final_15_items].dropna() 
print(f"N = {data_18.shape[0]}\n")

def tetrachoric_pair(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    p1 = np.mean(x); p2 = np.mean(y); p11 = np.mean((x == 1) & (y == 1))
    eps = 1e-8
    p1 = np.clip(p1, eps, 1-eps); p2 = np.clip(p2, eps, 1-eps); p11 = np.clip(p11, eps, 1-eps)
    h = -stats.norm.ppf(p1); k = -stats.norm.ppf(p2)
    def bvn_cdf(rho):
        cov = [[1, rho], [rho, 1]]
        mv = stats.multivariate_normal(mean=[0,0], cov=cov)
        return mv.cdf([-h, -k])
    lo, hi = -0.9999, 0.9999
    for _ in range(60):
        mid = (lo+hi)/2
        if bvn_cdf(mid) < p11: lo = mid
        else: hi = mid
    rho = (lo+hi)/2
    return np.clip(rho, -1+eps, 1-eps)

def tetrachoric_matrix(df):
    items = df.columns.tolist()
    n = len(items)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i+1, n):
            r = tetrachoric_pair(df.iloc[:,i].values, df.iloc[:,j].values)
            mat[i,j] = mat[j,i] = r
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.clip(eigvals, 1e-6, None)
    mat_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(mat_psd))
    mat_psd = mat_psd / np.outer(d, d)
    return pd.DataFrame(mat_psd, index=items, columns=items)

print("Computing tetrachoric correlations for 18 items...")
corr_18 = tetrachoric_matrix(data_18)
corr_15 = corr_18.loc[final_15_items, final_15_items] 


chi2, p = calculate_bartlett_sphericity(data_18)
kmo_all, kmo_total = calculate_kmo(data_18)
print(f"Bartlett's test: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"KMO total = {kmo_total:.3f}\n")

fa1 = FactorAnalyzer(n_factors=1, rotation=None, method='minres', is_corr_matrix=True)
fa1.fit(corr_18.values)
loadings_18 = fa1.loadings_.flatten()
print("--- Factor loadings (1-factor, 18 items) ---")
for item, l in zip(all_18_items, loadings_18):
    short = item.replace('pre_what_you_experiencing_during_covid_period_','')
    print(f"{short:55s} {l:+.3f} {' *' if abs(l)<0.30 else ''}")
ev18, _ = fa1.get_eigenvalues()
print(f"\nEigenvalue (18 items): {ev18[0]:.3f}  (% var = {ev18[0]/18*100:.1f}%)\n")

fa_15 = FactorAnalyzer(n_factors=1, rotation=None, method='minres', is_corr_matrix=True)
fa_15.fit(corr_15.values)
loadings_15 = fa_15.loadings_.flatten()
print("--- Factor loadings (1-factor, final 15 items) ---")
for item, l in zip(final_15_items, loadings_15):
    short = item.replace('pre_what_you_experiencing_during_covid_period_','')
    print(f"{short:55s} {l:+.3f}")
ev15, _ = fa_15.get_eigenvalues()
print(f"\nEigenvalue (15 items): {ev15[0]:.3f}  (% var = {ev15[0]/15*100:.1f}%)\n")


def alpha(df):
    item_var = df.var(axis=0, ddof=1)
    total_var = df.sum(axis=1).var(ddof=1)
    n = df.shape[1]
    return (n/(n-1)) * (1 - item_var.sum()/total_var)

alpha_18 = alpha(data_18)
alpha_15 = alpha(data_15)
print(f"Cronbach's alpha (18 items): {alpha_18:.3f}")
print(f"Cronbach's alpha (15 items): {alpha_15:.3f}\n")

print("--- Alpha if item dropped (18 items) ---")
for item in all_18_items:
    alpha_drop = alpha(data_18.drop(columns=[item]))
    short = item.replace('pre_what_you_experiencing_during_covid_period_','')
    flag = "  ↑ improves" if alpha_drop > alpha_18 else ""
    print(f"  Drop {short[:50]:50s} alpha = {alpha_drop:.3f}{flag}")


print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
print(f"KMO = {kmo_total:.3f}")
print(f"Bartlett's test p < 0.001\n")
print("18‑item set (all original):")
print(f"  - Items with low loadings (<0.30): 'difficulty_washing_hand' (0.09), 'difficulty_obtaining_mask' (0.34) – but financial support had negative -0.99")
print(f"  - Cronbach's α = {alpha_18:.3f}")
print(f"  - Eigenvalue = {ev18[0]:.2f}, variance = {ev18[0]/18*100:.1f}%")
print("\nFinal 15‑item set (your choice):")
print(f"  - Excludes financial support, mask, handwashing – includes emotional support")
print(f"  - All loadings > 0.40 (emotional support = {loadings_15[12]:.3f})")
print(f"  - Cronbach's α = {alpha_15:.3f}  (higher than 18-item set)")
print(f"  - Eigenvalue = {ev15[0]:.2f}, variance = {ev15[0]/15*100:.1f}%")
print(f"\n→ Improvement: α +{alpha_15 - alpha_18:.3f}, variance +{(ev15[0]/15*100) - (ev18[0]/18*100):.1f}%")
print("="*60)

N = 338

Computing tetrachoric correlations for 18 items...
Bartlett's test: χ² = 1047.17, p = 0.0000
KMO total = 0.785

--- Factor loadings (1-factor, 18 items) ---
anxiety                                                 +0.655 
depression                                              +0.569 
loneliness                                              +0.674 
anger                                                   +0.702 
frustration                                             +0.685 
grief_ofFeeling_ofLoss                                  +0.711 
changing_sleep_patern                                   +0.658 
notGetting_exercise                                     +0.545 
fear_ofGetting_covid                                    +0.433 
fear_ofGiving_covid                                     +0.591 
worrying_about_people_otherThanMe                       +0.610 
stigma_from_other_people                                +0.650 
notGetting_enough_financial_support                     -0.988 
co

In [5]:
all_18_items_post = [
    'what_you_experiencing_during_covid_period_anxiety',
    'what_you_experiencing_during_covid_period_depression',
    'what_you_experiencing_during_covid_period_loneliness',
    'what_you_experiencing_during_covid_period_anger',
    'what_you_experiencing_during_covid_period_frustration',
    'what_you_experiencing_during_covid_period_grief_ofFeeling_ofLoss',
    'what_you_experiencing_during_covid_period_changing_sleep_patern',
    'what_you_experiencing_during_covid_period_notGetting_exercise',
    'what_you_experiencing_during_covid_period_fear_ofGetting_covid',
    'what_you_experiencing_during_covid_period_fear_ofGiving_covid',
    'what_you_experiencing_during_covid_period_worrying_about_people_otherThanMe',
    'what_you_experiencing_during_covid_period_stigma_from_other_people',
    'what_you_experiencing_during_covid_period_notGetting_enough_financial_support',
    'what_you_experiencing_during_covid_period_confusion_about_covid',
    'what_you_experiencing_during_covid_period_confusion_about_where_toGet_info_about_covid',
    'what_you_experiencing_during_covid_period_notGetting_enough_emotionOrSocial_support',
    'what_you_experiencing_during_covid_period_difficulty_obtaining_mask',
    'what_you_experiencing_during_covid_period_difficulty_washing_hand'
]

final_15_items_post = [
    'what_you_experiencing_during_covid_period_anxiety',
    'what_you_experiencing_during_covid_period_depression',
    'what_you_experiencing_during_covid_period_loneliness',
    'what_you_experiencing_during_covid_period_anger',
    'what_you_experiencing_during_covid_period_frustration',
    'what_you_experiencing_during_covid_period_grief_ofFeeling_ofLoss',
    'what_you_experiencing_during_covid_period_changing_sleep_patern',
    'what_you_experiencing_during_covid_period_notGetting_exercise',
    'what_you_experiencing_during_covid_period_fear_ofGetting_covid',
    'what_you_experiencing_during_covid_period_fear_ofGiving_covid',
    'what_you_experiencing_during_covid_period_worrying_about_people_otherThanMe',
    'what_you_experiencing_during_covid_period_stigma_from_other_people',
    'what_you_experiencing_during_covid_period_notGetting_enough_emotionOrSocial_support',
    'what_you_experiencing_during_covid_period_confusion_about_covid',
    'what_you_experiencing_during_covid_period_confusion_about_where_toGet_info_about_covid'
]

data_18_post = data_post[all_18_items_post].dropna()
data_15_post = data_post[final_15_items_post].dropna()
print(f"N = {data_18_post.shape[0]}\n")


def tetrachoric_pair(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    p1 = np.mean(x); p2 = np.mean(y); p11 = np.mean((x == 1) & (y == 1))
    eps = 1e-8
    p1 = np.clip(p1, eps, 1-eps); p2 = np.clip(p2, eps, 1-eps); p11 = np.clip(p11, eps, 1-eps)
    h = -stats.norm.ppf(p1); k = -stats.norm.ppf(p2)
    def bvn_cdf(rho):
        cov = [[1, rho], [rho, 1]]
        mv = stats.multivariate_normal(mean=[0,0], cov=cov)
        return mv.cdf([-h, -k])
    lo, hi = -0.9999, 0.9999
    for _ in range(60):
        mid = (lo+hi)/2
        if bvn_cdf(mid) < p11: lo = mid
        else: hi = mid
    rho = (lo+hi)/2
    return np.clip(rho, -1+eps, 1-eps)

def tetrachoric_matrix(df):
    items = df.columns.tolist()
    n = len(items)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i+1, n):
            r = tetrachoric_pair(df.iloc[:,i].values, df.iloc[:,j].values)
            mat[i,j] = mat[j,i] = r
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.clip(eigvals, 1e-6, None)
    mat_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(mat_psd))
    mat_psd = mat_psd / np.outer(d, d)
    return pd.DataFrame(mat_psd, index=items, columns=items)

print("Computing tetrachoric correlations for 18 items (post)...")
corr_18_post = tetrachoric_matrix(data_18_post)
corr_15_post = corr_18_post.loc[final_15_items_post, final_15_items_post]


chi2, p = calculate_bartlett_sphericity(data_18_post)
kmo_all, kmo_total = calculate_kmo(data_18_post)
print(f"Bartlett's test: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"KMO total = {kmo_total:.3f}\n")

fa1 = FactorAnalyzer(n_factors=1, rotation=None, method='minres', is_corr_matrix=True)
fa1.fit(corr_18_post.values)
loadings_18 = fa1.loadings_.flatten()
print("--- Factor loadings (1-factor, 18 items, post) ---")
for item, l in zip(all_18_items_post, loadings_18):
    short = item.replace('what_you_experiencing_during_covid_period_','')
    print(f"{short:55s} {l:+.3f} {' *' if abs(l)<0.30 else ''}")
ev18, _ = fa1.get_eigenvalues()
print(f"\nEigenvalue (18 items): {ev18[0]:.3f}  (% var = {ev18[0]/18*100:.1f}%)\n")

fa_15 = FactorAnalyzer(n_factors=1, rotation=None, method='minres', is_corr_matrix=True)
fa_15.fit(corr_15_post.values)
loadings_15 = fa_15.loadings_.flatten()
print("--- Factor loadings (1-factor, final 15 items, post) ---")
for item, l in zip(final_15_items_post, loadings_15):
    short = item.replace('what_you_experiencing_during_covid_period_','')
    print(f"{short:55s} {l:+.3f}")
ev15, _ = fa_15.get_eigenvalues()
print(f"\nEigenvalue (15 items): {ev15[0]:.3f}  (% var = {ev15[0]/15*100:.1f}%)\n")


def alpha(df):
    item_var = df.var(axis=0, ddof=1)
    total_var = df.sum(axis=1).var(ddof=1)
    n = df.shape[1]
    return (n/(n-1)) * (1 - item_var.sum()/total_var)

alpha_18 = alpha(data_18_post)
alpha_15 = alpha(data_15_post)
print(f"Cronbach's alpha (18 items): {alpha_18:.3f}")
print(f"Cronbach's alpha (15 items): {alpha_15:.3f}\n")

print("--- Alpha if item dropped (18 items, post) ---")
for item in all_18_items_post:
    alpha_drop = alpha(data_18_post.drop(columns=[item]))
    short = item.replace('what_you_experiencing_during_covid_period_','')
    flag = "  ↑ improves" if alpha_drop > alpha_18 else ""
    print(f"  Drop {short[:50]:50s} alpha = {alpha_drop:.3f}{flag}")


print("\n" + "="*60)
print("POST‑WAVE COMPARISON SUMMARY")
print("="*60)
print(f"KMO = {kmo_total:.3f}")
print(f"Bartlett's test p < 0.001\n")
print("18‑item set (all original):")
print(f"  - Low loadings (<0.30): likely 'difficulty_washing_hand' etc.")
print(f"  - Cronbach's α = {alpha_18:.3f}")
print(f"  - Eigenvalue = {ev18[0]:.2f}, variance = {ev18[0]/18*100:.1f}%")
print("\nFinal 15‑item set (your composite):")
print(f"  - Excludes financial support, mask, handwashing – includes emotional support")
print(f"  - All loadings > 0.40 (emotional support loading = {loadings_15[12]:.3f})")
print(f"  - Cronbach's α = {alpha_15:.3f}")
print(f"  - Eigenvalue = {ev15[0]:.2f}, variance = {ev15[0]/15*100:.1f}%")
print(f"\n→ Improvement: α +{alpha_15 - alpha_18:.3f}, variance +{(ev15[0]/15*100) - (ev18[0]/18*100):.1f}%")
print("="*60)

N = 338

Computing tetrachoric correlations for 18 items (post)...
Bartlett's test: χ² = 1434.14, p = 0.0000
KMO total = 0.850

--- Factor loadings (1-factor, 18 items, post) ---
anxiety                                                 +0.793 
depression                                              +0.710 
loneliness                                              +0.719 
anger                                                   +0.767 
frustration                                             +0.762 
grief_ofFeeling_ofLoss                                  +0.735 
changing_sleep_patern                                   +0.720 
notGetting_exercise                                     +0.617 
fear_ofGetting_covid                                    +0.573 
fear_ofGiving_covid                                     +0.634 
worrying_about_people_otherThanMe                       +0.732 
stigma_from_other_people                                +0.549 
notGetting_enough_financial_support                  

# PTSS

In [6]:
ptss_items_pre = [
    'pre_repeated_disturbing_dream_ofStressfull_experiencce_from_past',
    'pre_repeated_disturbing_memory_though_image_ofStressfull_experiencce_from_past',
    'pre_suddenly_feel_asIf_aStressfull_experience_were_happening_again',
    'pre_feeling_upset_when_something_remind_you_ofStressful_experience_fromPast',
    'pre_avoiding_activy_because_they_remind_you_stressful_experience_from_past',
    'pre_having_physical_reaction_when_somthing_remind_you_stressful_experience',
    'pre_avoiding_thinking_about_stressful_experience_from_past',
    'pre_loss_ofInterest_activity_that_you_enjoyed',
    'pre_feeling_emotional_numb_orBeeing_unble_toLoving_feeling_forThose_closeTo_you',
    'pre_feeling_asIfYourFeature_willBer_cut_short',
    'pre_having_difficulty_concentrating',
    'pre_feeling_irretablee_orHaving_angry_outburst',
    'pre_trouble_falling_orStaying_asSleep',
    'pre_trouble_memembering_important_part_ofStressful_experience_from_past',
    'pre_feeling_distant_from_other_people'
]

data_ptss = data_pre[ptss_items_pre].dropna()
print(f"N = {data_ptss.shape[0]}\n")

def tetrachoric_pair(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    p1 = np.mean(x); p2 = np.mean(y); p11 = np.mean((x == 1) & (y == 1))
    eps = 1e-8
    p1 = np.clip(p1, eps, 1-eps); p2 = np.clip(p2, eps, 1-eps); p11 = np.clip(p11, eps, 1-eps)
    h = -stats.norm.ppf(p1); k = -stats.norm.ppf(p2)
    def bvn_cdf(rho):
        cov = [[1, rho], [rho, 1]]
        mv = stats.multivariate_normal(mean=[0,0], cov=cov)
        return mv.cdf([-h, -k])
    lo, hi = -0.9999, 0.9999
    for _ in range(60):
        mid = (lo+hi)/2
        if bvn_cdf(mid) < p11: lo = mid
        else: hi = mid
    rho = (lo+hi)/2
    return np.clip(rho, -1+eps, 1-eps)

def tetrachoric_matrix(df):
    items = df.columns.tolist()
    n = len(items)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i+1, n):
            r = tetrachoric_pair(df.iloc[:,i].values, df.iloc[:,j].values)
            mat[i,j] = mat[j,i] = r
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.clip(eigvals, 1e-6, None)
    mat_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(mat_psd))
    mat_psd = mat_psd / np.outer(d, d)
    return pd.DataFrame(mat_psd, index=items, columns=items)

print("Computing tetrachoric correlations...")
corr_ptss = tetrachoric_matrix(data_ptss)

chi2, p = calculate_bartlett_sphericity(data_ptss)
kmo_all, kmo_total = calculate_kmo(data_ptss)
print(f"Bartlett's test: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"KMO total = {kmo_total:.3f}\n")


fa = FactorAnalyzer(
    n_factors=1,
    rotation=None,
    method='minres'
)
fa.fit(data_ptss)

loadings = fa.loadings_.flatten()


print("--- Factor loadings (1-factor, 15 PTSS items) ---")
for item, l in zip(ptss_items_pre, loadings):
    short = item.replace('pre_', '')
    print(f"{short:55s} {l:+.3f}")

ev, _ = fa.get_eigenvalues()
print(f"\nEigenvalue (15 items): {ev[0]:.3f}  (% var = {ev[0]/15*100:.1f}%)\n")


def alpha(df):
    item_var = df.var(axis=0, ddof=1)
    total_var = df.sum(axis=1).var(ddof=1)
    n = df.shape[1]
    return (n/(n-1)) * (1 - item_var.sum() / total_var)

alpha_ptss = alpha(data_ptss)
print(f"Cronbach's alpha (15 items): {alpha_ptss:.3f}")

print("\n--- Alpha if item dropped ---")
for item in ptss_items_pre:
    alpha_drop = alpha(data_ptss.drop(columns=[item]))
    short = item.replace('pre_', '')
    flag = "  ↑ improves" if alpha_drop > alpha_ptss else ""
    print(f"  Drop {short[:50]:50s} alpha = {alpha_drop:.3f}{flag}")

print("\n" + "="*60)
print("PTSS – FACTOR ANALYSIS SUMMARY (Pre‑wave)")
print("="*60)
print(f"KMO = {kmo_total:.3f}")
print(f"Bartlett's test p < 0.001\n")
print(f"Cronbach's α = {alpha_ptss:.3f}")
print(f"Eigenvalue = {ev[0]:.2f}, variance = {ev[0]/15*100:.1f}%")
print(f"Loadings range: {loadings.min():.3f} to {loadings.max():.3f}")
print("="*60)

N = 338

Computing tetrachoric correlations...
Bartlett's test: χ² = 3409.57, p = 0.0000
KMO total = 0.943

--- Factor loadings (1-factor, 15 PTSS items) ---
repeated_disturbing_dream_ofStressfull_experiencce_from_past +0.702
repeated_disturbing_memory_though_image_ofStressfull_experiencce_from_past +0.735
suddenly_feel_asIf_aStressfull_experience_were_happening_again +0.744
feeling_upset_when_something_remind_you_ofStressful_experience_fromPast +0.812
avoiding_activy_because_they_remind_you_stressful_experience_from_past +0.721
having_physical_reaction_when_somthing_remind_you_stressful_experience +0.705
avoiding_thinking_about_stressful_experience_from_past  +0.743
loss_ofInterest_activity_that_you_enjoyed               +0.769
feeling_emotional_numb_orBeeing_unble_toLoving_feeling_forThose_closeTo_you +0.743
feeling_asIfYourFeature_willBer_cut_short               +0.714
having_difficulty_concentrating                         +0.768
feeling_irretablee_orHaving_angry_outburst          

In [7]:
ptss_items_post = [
    'repeated_disturbing_dream_ofStressfull_experiencce_from_past',
    'repeated_disturbing_memory_though_image_ofStressfull_experiencce_from_past',
    'suddenly_feel_asIf_aStressfull_experience_were_happening_again',
    'feeling_upset_when_something_remind_you_ofStressful_experience_fromPast',
    'having_physical_reaction_when_somthing_remind_you_stressful_experience',
    'avoiding_activy_because_they_remind_you_stressful_experience_from_past',
    'avoiding_thinking_about_stressful_experience_from_past',
    'trouble_memembering_important_part_ofStressful_experience_from_past',
    'loss_ofInterest_activity_that_you_enjoyed',
    'feeling_distant_from_other_people',
    'feeling_emotional_numb_orBeeing_unble_toLoving_feeling_forThose_closeTo_you',
    'feeling_asIfYourFeature_willBer_cut_short',
    'having_difficulty_concentrating',
    'feeling_irretablee_orHaving_angry_outburst',
    'trouble_falling_orStaying_asSleep'
]

data_ptss_post = data_post[ptss_items_post].dropna()
print(f"N = {data_ptss_post.shape[0]}\n")

def tetrachoric_pair(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    p1 = np.mean(x); p2 = np.mean(y); p11 = np.mean((x == 1) & (y == 1))
    eps = 1e-8
    p1 = np.clip(p1, eps, 1-eps); p2 = np.clip(p2, eps, 1-eps); p11 = np.clip(p11, eps, 1-eps)
    h = -stats.norm.ppf(p1); k = -stats.norm.ppf(p2)
    def bvn_cdf(rho):
        cov = [[1, rho], [rho, 1]]
        mv = stats.multivariate_normal(mean=[0,0], cov=cov)
        return mv.cdf([-h, -k])
    lo, hi = -0.9999, 0.9999
    for _ in range(60):
        mid = (lo+hi)/2
        if bvn_cdf(mid) < p11: lo = mid
        else: hi = mid
    rho = (lo+hi)/2
    return np.clip(rho, -1+eps, 1-eps)

def tetrachoric_matrix(df):
    items = df.columns.tolist()
    n = len(items)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i+1, n):
            r = tetrachoric_pair(df.iloc[:,i].values, df.iloc[:,j].values)
            mat[i,j] = mat[j,i] = r
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.clip(eigvals, 1e-6, None)
    mat_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(mat_psd))
    mat_psd = mat_psd / np.outer(d, d)
    return pd.DataFrame(mat_psd, index=items, columns=items)

print("Computing tetrachoric correlations (post‑wave)...")
corr_ptss_post = tetrachoric_matrix(data_ptss_post)

chi2, p = calculate_bartlett_sphericity(data_ptss_post)
kmo_all, kmo_total = calculate_kmo(data_ptss_post)
print(f"Bartlett's test: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"KMO total = {kmo_total:.3f}\n")

fa = FactorAnalyzer(n_factors=1, rotation=None, method='minres', is_corr_matrix=True)
fa.fit(corr_ptss_post.values)
loadings = fa.loadings_.flatten()

print("--- Factor loadings (1-factor, 15 PTSS items, post) ---")
for item, l in zip(ptss_items_post, loadings):
    print(f"{item:55s} {l:+.3f}")

ev, _ = fa.get_eigenvalues()
print(f"\nEigenvalue (15 items): {ev[0]:.3f}  (% var = {ev[0]/15*100:.1f}%)\n")

def alpha(df):
    item_var = df.var(axis=0, ddof=1)
    total_var = df.sum(axis=1).var(ddof=1)
    n = df.shape[1]
    return (n/(n-1)) * (1 - item_var.sum() / total_var)

alpha_ptss_post = alpha(data_ptss_post)
print(f"Cronbach's alpha (15 items): {alpha_ptss_post:.3f}")

print("\n--- Alpha if item dropped (post) ---")
for item in ptss_items_post:
    alpha_drop = alpha(data_ptss_post.drop(columns=[item]))
    flag = "  ↑ improves" if alpha_drop > alpha_ptss_post else ""
    print(f"  Drop {item[:50]:50s} alpha = {alpha_drop:.3f}{flag}")

print("\n" + "="*60)
print("PTSS – FACTOR ANALYSIS SUMMARY (Post‑wave)")
print("="*60)
print(f"KMO = {kmo_total:.3f}")
print(f"Bartlett's test p < 0.001\n")
print(f"Cronbach's α = {alpha_ptss_post:.3f}")
print(f"Eigenvalue = {ev[0]:.2f}, variance = {ev[0]/15*100:.1f}%")
print(f"Loadings range: {loadings.min():.3f} to {loadings.max():.3f}")
print("="*60)

N = 338

Computing tetrachoric correlations (post‑wave)...
Bartlett's test: χ² = 3898.29, p = 0.0000
KMO total = 0.948

--- Factor loadings (1-factor, 15 PTSS items, post) ---
repeated_disturbing_dream_ofStressfull_experiencce_from_past +0.037
repeated_disturbing_memory_though_image_ofStressfull_experiencce_from_past +0.037
suddenly_feel_asIf_aStressfull_experience_were_happening_again +0.037
feeling_upset_when_something_remind_you_ofStressful_experience_fromPast -1.006
having_physical_reaction_when_somthing_remind_you_stressful_experience +0.037
avoiding_activy_because_they_remind_you_stressful_experience_from_past +0.037
avoiding_thinking_about_stressful_experience_from_past  +0.037
trouble_memembering_important_part_ofStressful_experience_from_past +0.037
loss_ofInterest_activity_that_you_enjoyed               +0.037
feeling_distant_from_other_people                       +0.037
feeling_emotional_numb_orBeeing_unble_toLoving_feeling_forThose_closeTo_you +0.037
feeling_asIfYourFeatur

# coping

In [8]:
coping_items_pre = [
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_onPhone',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_video_chat',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_inPerson',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_inHome',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_outdoor',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_gardening',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_meditation',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_creative_activity',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_learn_new_skill',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taking_breaks_from_the_news',
    'pre_what_things_you_done_toTakeCare_ofMentalHealth_during_covid_spent_time_with_pet'
]

data_coping_pre = data_pre[coping_items_pre].dropna()
print(f"Pre‑wave coping: N = {data_coping_pre.shape[0]}\n")

def tetrachoric_pair(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    p1 = np.mean(x); p2 = np.mean(y); p11 = np.mean((x == 1) & (y == 1))
    eps = 1e-8
    p1 = np.clip(p1, eps, 1-eps); p2 = np.clip(p2, eps, 1-eps); p11 = np.clip(p11, eps, 1-eps)
    h = -stats.norm.ppf(p1); k = -stats.norm.ppf(p2)
    def bvn_cdf(rho):
        cov = [[1, rho], [rho, 1]]
        mv = stats.multivariate_normal(mean=[0,0], cov=cov)
        return mv.cdf([-h, -k])
    lo, hi = -0.9999, 0.9999
    for _ in range(60):
        mid = (lo+hi)/2
        if bvn_cdf(mid) < p11: lo = mid
        else: hi = mid
    rho = (lo+hi)/2
    return np.clip(rho, -1+eps, 1-eps)

def tetrachoric_matrix(df):
    items = df.columns.tolist()
    n = len(items)
    mat = np.eye(n)
    for i in range(n):
        for j in range(i+1, n):
            r = tetrachoric_pair(df.iloc[:,i].values, df.iloc[:,j].values)
            mat[i,j] = mat[j,i] = r
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.clip(eigvals, 1e-6, None)
    mat_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(mat_psd))
    mat_psd = mat_psd / np.outer(d, d)
    return pd.DataFrame(mat_psd, index=items, columns=items)

print("Computing tetrachoric correlations (pre‑wave coping)...")
corr_coping_pre = tetrachoric_matrix(data_coping_pre)

chi2, p = calculate_bartlett_sphericity(data_coping_pre)
kmo_all, kmo_total = calculate_kmo(data_coping_pre)
print(f"Bartlett's test: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"KMO total = {kmo_total:.3f}\n")


fa = FactorAnalyzer(n_factors=1, rotation=None, method='minres', is_corr_matrix=True)
fa.fit(corr_coping_pre.values)
loadings = fa.loadings_.flatten()

print("--- Factor loadings (1‑factor, 11 coping items, pre) ---")
for item, l in zip(coping_items_pre, loadings):
    short = item.replace('pre_', '')
    print(f"{short:55s} {l:+.3f}")

ev, _ = fa.get_eigenvalues()
print(f"\nEigenvalue: {ev[0]:.3f}  (% var = {ev[0]/11*100:.1f}%)\n")


def alpha(df):
    item_var = df.var(axis=0, ddof=1)
    total_var = df.sum(axis=1).var(ddof=1)
    n = df.shape[1]
    return (n/(n-1)) * (1 - item_var.sum() / total_var)

alpha_coping_pre = alpha(data_coping_pre)
print(f"Cronbach's alpha (11 items): {alpha_coping_pre:.3f}")

print("\n--- Alpha if item dropped (pre) ---")
for item in coping_items_pre:
    alpha_drop = alpha(data_coping_pre.drop(columns=[item]))
    short = item.replace('pre_', '')
    flag = "  ↑ improves" if alpha_drop > alpha_coping_pre else ""
    print(f"  Drop {short[:50]:50s} alpha = {alpha_drop:.3f}{flag}")

print("\n" + "="*60)
print("COPING STRATEGY – FACTOR ANALYSIS SUMMARY (Pre‑wave)")
print("="*60)
print(f"KMO = {kmo_total:.3f}")
print(f"Bartlett's test p < 0.001\n")
print(f"Cronbach's α = {alpha_coping_pre:.3f}")
print(f"Eigenvalue = {ev[0]:.2f}, variance = {ev[0]/11*100:.1f}%")
print(f"Loadings range: {loadings.min():.3f} to {loadings.max():.3f}")
print("="*60)

Pre‑wave coping: N = 338

Computing tetrachoric correlations (pre‑wave coping)...
Bartlett's test: χ² = 353.40, p = 0.0000
KMO total = 0.740

--- Factor loadings (1‑factor, 11 coping items, pre) ---
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_onPhone -0.531
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_video_chat -0.654
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_inPerson -0.427
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_inHome -0.569
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_outdoor -0.433
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_gardening -0.554
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_meditation -0.542
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_creative_activity -0.518
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_learn_new_skill -0.505
what_things_you_done_toTakeCare_ofMentalHealth

In [9]:
coping_items_post = [
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_onPhone',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_video_chat',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_inPerson',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_inHome',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_outdoor',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_gardening',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_meditation',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_creative_activity',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_learn_new_skill',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taking_breaks_from_the_news',
    'what_things_you_done_toTakeCare_ofMentalHealth_during_covid_spent_time_with_pet'
]

data_coping_post = data_post[coping_items_post].dropna()
print(f"Post‑wave coping: N = {data_coping_post.shape[0]}\n")

print("Computing tetrachoric correlations (post‑wave coping)...")
corr_coping_post = tetrachoric_matrix(data_coping_post)

chi2, p = calculate_bartlett_sphericity(data_coping_post)
kmo_all, kmo_total = calculate_kmo(data_coping_post)
print(f"Bartlett's test: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"KMO total = {kmo_total:.3f}\n")

fa = FactorAnalyzer(n_factors=1, rotation=None, method='minres', is_corr_matrix=True)
fa.fit(corr_coping_post.values)
loadings = fa.loadings_.flatten()

print("--- Factor loadings (1‑factor, 11 coping items, post) ---")
for item, l in zip(coping_items_post, loadings):
    print(f"{item:55s} {l:+.3f}")

ev, _ = fa.get_eigenvalues()
print(f"\nEigenvalue: {ev[0]:.3f}  (% var = {ev[0]/11*100:.1f}%)\n")

alpha_coping_post = alpha(data_coping_post)
print(f"Cronbach's alpha (11 items): {alpha_coping_post:.3f}")

print("\n--- Alpha if item dropped (post) ---")
for item in coping_items_post:
    alpha_drop = alpha(data_coping_post.drop(columns=[item]))
    flag = "  ↑ improves" if alpha_drop > alpha_coping_post else ""
    print(f"  Drop {item[:50]:50s} alpha = {alpha_drop:.3f}{flag}")

print("\n" + "="*60)
print("COPING STRATEGY – FACTOR ANALYSIS SUMMARY (Post‑wave)")
print("="*60)
print(f"KMO = {kmo_total:.3f}")
print(f"Bartlett's test p < 0.001\n")
print(f"Cronbach's α = {alpha_coping_post:.3f}")
print(f"Eigenvalue = {ev[0]:.2f}, variance = {ev[0]/11*100:.1f}%")
print(f"Loadings range: {loadings.min():.3f} to {loadings.max():.3f}")
print("="*60)

Post‑wave coping: N = 338

Computing tetrachoric correlations (post‑wave coping)...
Bartlett's test: χ² = 806.12, p = 0.0000
KMO total = 0.845

--- Factor loadings (1‑factor, 11 coping items, post) ---
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_onPhone +0.837
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_video_chat +0.868
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_taked_toFNF_inPerson +0.685
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_inHome +0.756
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_exercise_outdoor +0.602
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_gardening +0.530
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_meditation +0.696
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_creative_activity +0.713
what_things_you_done_toTakeCare_ofMentalHealth_during_covid_learn_new_skill +0.632
what_things_you_done_toTakeCare_ofMentalHea